# Collaborative Filtering Recommender

Item-Based Collaborative Filtering recommends items by computing similarities between items based on historical user interactions (rather than user similarities). This approach is highly scalable because item relationships are more stable than user profiles. This notebook generates synthetic item purchase ratings (800 users across 120 products), calculates Cosine item similarity, predicts user ratings, and plots recommendation matrices.



In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity

# Generate synthetic ratings dataset (800 users, 120 products, sparse rating entries)
np.random.seed(192)
n_users = 800
n_items = 120

# Create sparse rating matrix (fill scale 1-5, sparsity ratio = 92%)
raw_ratings = np.random.choice([0, 1, 2, 3, 4, 5], size=(n_users, n_items), p=[0.92, 0.015, 0.015, 0.02, 0.015, 0.015])

df_ratings = pd.DataFrame(raw_ratings, columns=[f"Prod_{i+1:03d}" for i in range(n_items)])
df_ratings.index = [f"User_{i+1:03d}" for i in range(n_users)]
df_ratings.head(10)



,Prod_001,Prod_002,Prod_003,Prod_004,Prod_005,Prod_006,Prod_007,Prod_008,Prod_009,Prod_010,Prod_011,Prod_012,Prod_013,Prod_014,Prod_015,Prod_016,Prod_017,Prod_018,Prod_019,Prod_020,Prod_021,Prod_022,Prod_023,Prod_024,Prod_025,Prod_026,Prod_027,Prod_028,Prod_029,Prod_030,Prod_031,Prod_032,Prod_033,Prod_034,Prod_035,Prod_036,Prod_037,Prod_038,Prod_039,Prod_040,Prod_041,Prod_042,Prod_043,Prod_044,Prod_045,Prod_046,Prod_047,Prod_048,Prod_049,Prod_050,Prod_051,Prod_052,Prod_053,Prod_054,Prod_055,Prod_056,Prod_057,Prod_058,Prod_059,Prod_060,Prod_061,Prod_062,Prod_063,Prod_064,Prod_065,Prod_066,Prod_067,Prod_068,Prod_069,Prod_070,Prod_071,Prod_072,Prod_073,Prod_074,Prod_075,Prod_076,Prod_077,Prod_078,Prod_079,Prod_080,Prod_081,Prod_082,Prod_083,Prod_084,Prod_085,Prod_086,Prod_087,Prod_088,Prod_089,Prod_090,Prod_091,Prod_092,Prod_093,Prod_094,Prod_095,Prod_096,Prod_097,Prod_098,Prod_099,Prod_100,Prod_101,Prod_102,Prod_103,Prod_104,Prod_105,Prod_106,Prod_107,Prod_108,Prod_109,Prod_110,Prod_111,Prod_112,Prod_113,Prod_114,Prod_115,Prod_116,Prod_117,Prod_118,Prod_119,Prod_120
User_001,0,0,0,1,0,0,0,0,0,0,0,0,3,0,0,0,0,3,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,5,0,0,0,0,5,4,0,0,0,0,0,0,0,0,0,0
User_002,0,0,0,3,0,0,0,0,0,0,1,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,3,0,0,0,4,0,0,0,0,0,0,0,1,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,2,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,3,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
User_003,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,2,0,0,0,4,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,3,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5
User_004,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,3,5,0,1,0,5,0,0,0,2,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0
User_005,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,3,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,3,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0
User_006,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,4,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,4,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,2,0,3,0,0,0,0
User_007,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
User_008,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,3,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0
User_009,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,1,0,0,0,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,3,0,0,0,0,2,0,0,0,0,0,0,0,0,5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5
User_010,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,4,0,0,0,0,0,0,0


## Item Similarity Matrix Calculation

We compute the pairwise Cosine similarity between product columns, using mean-centered ratings to handle varying user bias.



In [2]:
# Centering ratings by subtracting user means (for items)
ratings_centered = df_ratings.copy()
for col in df_ratings.columns:
    col_mean = df_ratings[col][df_ratings[col] > 0].mean()
    if pd.isna(col_mean):
        col_mean = 0.0
    # Center only non-zero entries
    ratings_centered[col] = df_ratings[col].apply(lambda x: x - col_mean if x > 0 else 0.0)

# Compute pairwise item similarity matrix: shape [120 items, 120 items]
item_similarity = cosine_similarity(ratings_centered.T)
np.fill_diagonal(item_similarity, 0.0) # Zero self similarity

df_sim = pd.DataFrame(item_similarity, index=df_ratings.columns, columns=df_ratings.columns)
print("Item Cosine Similarity Matrix Summary:")
df_sim.iloc[:8, :8]



Item Cosine Similarity Matrix Summary:


,Prod_001,Prod_002,Prod_003,Prod_004,Prod_005,Prod_006,Prod_007,Prod_008
Prod_001,0.000000,-0.019291,-0.008334,-0.007158,0.012918,0.025334,-0.033149,-0.007991
Prod_002,-0.019291,0.000000,-0.015701,-0.000471,0.037323,-0.017855,-0.040573,0.009762
Prod_003,-0.008334,-0.015701,0.000000,-0.059266,-0.028777,-0.001590,-0.031211,-0.041610
Prod_004,-0.007158,-0.000471,-0.059266,0.000000,-0.008570,-0.034731,0.059633,-0.064482
Prod_005,0.012918,0.037323,-0.028777,-0.008570,0.000000,-0.004188,0.038975,0.005921
Prod_006,0.025334,-0.017855,-0.001590,-0.034731,-0.004188,0.000000,0.027589,0.061608
Prod_007,-0.033149,-0.040573,-0.031211,0.059633,0.038975,0.027589,0.000000,0.035643
Prod_008,-0.007991,0.009762,-0.041610,-0.064482,0.005921,0.061608,0.035643,0.000000


## Collaborative Recommendations Matrix Heatmap

Using Plotly, we visualize the item-to-item similarity scores between the first 30 products to reveal item correlation clusters.



In [3]:
# Convert matrix to standard lists for JSON stability
fig = go.Figure(go.Heatmap(
    z=item_similarity[:30, :30].tolist(),
    x=list(df_ratings.columns[:30]),
    y=list(df_ratings.columns[:30]),
    colorscale='Cividis',
    colorbar=dict(title="Cosine Sim")
))

fig.update_layout(
    title='Item-to-Item Cosine Similarity Heatmap (First 30 Products)',
    xaxis_title='Product Catalog Code',
    yaxis_title='Product Catalog Code',
    width=650,
    height=550,
    template='plotly_white'
)

fig.show()
